In [ ]:
import os
import pathlib
import pandas as pd
from dotenv import dotenv_values
from sqlalchemy import text, create_engine

In [ ]:
root_path = pathlib.Path(os.getcwd())
root_path = root_path.parents[1]

In [ ]:
config = dotenv_values(root_path / ".env.local")

engine = create_engine(
    f"mysql+pymysql://{config['DB_USER']}:{config['DB_PASSWORD']}@{config['DB_HOST']}:{config['DB_PORT']}/{config['DB_NAME']}"
)

connection_database = engine.connect()

In [ ]:
dataframe = pd.read_excel(
    os.path.join(
        root_path,
        "data_folder",
        "product_detail_export_cleaned.xlsx",
    )
)

# Table `Distributor`
> - `id_distributor` (primary key, auto-increment)
> - `distributor_name` (string, not null)

In [ ]:
select_distributor = text("SELECT * FROM distributor")
database_distributor = pd.read_sql(select_distributor, connection_database)

In [ ]:
dataframe_distributor = pd.Series(dataframe["distributor"].unique(), name="distributor_name")

In [ ]:
insert_distributor = text(
    "INSERT INTO distributor (distributor_name) VALUES (:distributor_name)"
)

for distributor_name in dataframe_distributor:
    connection_database.execute(insert_distributor, {"distributor_name": distributor_name})

connection_database.commit()

# TABLE `Industrial`

> - `id_industrial` (primary key, auto-increment)
> - `industrial_name` (string, not null)

In [ ]:
select_industrial = text("SELECT * FROM industrial")
database_industrial = pd.read_sql(select_industrial, connection_database)

In [ ]:
dataframe_industrial = pd.Series(dataframe["industrial"].unique(), name="industrial_name")

In [ ]:
insert_industrial = text(
    "INSERT INTO industrial (industrial_name) VALUES (:industrial_name)"
)

for industrial_name in dataframe_industrial:
    connection_database.execute(insert_industrial, {"industrial_name": industrial_name})

connection_database.commit()

# Table `Brand`

> - `id_brand` (primary key, auto-increment)
> - `brand_name` (string, not null)

In [ ]:
select_brand = text("SELECT * FROM brand")
database_brand = pd.read_sql(select_brand, connection_database)

In [ ]:
dataframe_brand = pd.Series(dataframe["brand"].unique(), name="brand_name")

In [ ]:
insert_brand = text(
    "INSERT INTO brand (brand_name) VALUES (:brand_name)"
)

for brand_name in dataframe_brand:
    connection_database.execute(insert_brand, {"brand_name": brand_name})

connection_database.commit()

# Table `Unit`

> - `id_unit` (primary key, auto-increment)
> - `unit_name` (string, not null)

In [ ]:
select_unit = text("SELECT * FROM unit")
database_unit = pd.read_sql(select_unit, connection_database)

In [ ]:
dataframe_unit = pd.Series(dataframe["unit"].unique(), name="unit_name")

In [ ]:
insert_unit = text(
    "INSERT INTO unit (unit_name) VALUES (:unit_name)"
)

for unit_name in dataframe_unit:
    connection_database.execute(insert_unit, {"unit_name": unit_name})

connection_database.commit()

# Table `Data Source`

> - `id_data_source` (primary key, auto-increment)
> - `data_source_name` (string, not null)

In [ ]:
select_data_source = text("SELECT * FROM data_source")
database_data_source = pd.read_sql(select_data_source, connection_database)

In [ ]:
dataframe_data_source = pd.Series(dataframe["data_source"].unique(), name="data_source_name")

In [ ]:
insert_data_source = text(
    "INSERT INTO data_source (data_source_name) VALUES (:data_source_name)"
)

for data_source_name in dataframe_data_source:
    connection_database.execute(insert_data_source, {"data_source_name": data_source_name})

connection_database.commit()

# Table `Category`

> - `id_category` (primary key, auto-increment)
> - `category_name` (string, not null)

In [ ]:
select_category = text("SELECT * FROM category")
database_category = pd.read_sql(select_category, connection_database)

In [ ]:
mapping = pd.read_excel(
    os.path.join(root_path, "data_folder", "mapping", "mapping_product.xlsx"),
    sheet_name="mapping_products",
)

dataframe_category = pd.Series(mapping["categories"].unique(), name="category_name")

In [ ]:
insert_category = text(
    "INSERT INTO category (category_name) VALUES (:category_name)"
)

for category_name in dataframe_category:
    connection_database.execute(insert_category, {"category_name": category_name})

connection_database.commit()

# Table `Product`

> - `id_product` (primary key, auto-increment)
> - `fk_id_brand` → `brand.id_brand`
> - `fk_id_category` → `category.id_category`
> - `fk_id_unit` → `unit.id_unit`
> - `fk_id_data_source` → `data_source.id_data_source`
> - `product_name` (string, nullable)
> - `product_code` (string, nullable)
> - `description` (text, nullable)

In [ ]:
# Load final dataframe (product_name mapped) + reload reference tables with their DB ids
dataframe_final = pd.read_excel(
    os.path.join(root_path, "data_folder", "product_detail_export_final.xlsx")
)

database_brand       = pd.read_sql(text("SELECT * FROM brand"),       connection_database)
database_category    = pd.read_sql(text("SELECT * FROM category"),    connection_database)
database_unit        = pd.read_sql(text("SELECT * FROM unit"),        connection_database)
database_data_source = pd.read_sql(text("SELECT * FROM data_source"), connection_database)

# --- Verify if all brands in the final file exist in the database ---
brands_in_database  = set(database_brand["brand_name"])
brands_in_file = set(dataframe_final["brand"].dropna().unique())
missing_brands = brands_in_file - brands_in_database

if missing_brands:
    for brand in missing_brands:
        connection_database.execute(text("INSERT INTO brand (brand_name) VALUES (:name)"), {"name": brand})

    connection_database.commit()
    database_brand = pd.read_sql(text("SELECT * FROM brand"), connection_database)

# --- If any category in the final file is missing in DB, insert it (ex: "Non catégorisé") ---
if "Non catégorisé" not in set(database_category["category_name"]):
    connection_database.execute(
        text("INSERT INTO category (category_name) VALUES (:name)"),
        {"name": "Non catégorisé"},
    )
    connection_database.commit()
    database_category = pd.read_sql(text("SELECT * FROM category"), connection_database)

In [ ]:
dataframe_product = (
    dataframe_final
    [["product_name", "product_code", "description", "brand", "unit", "data_source"]]
    .copy() # copy for avoiding SettingWithCopyWarning in next steps (add new foreign keys)
)

# Step 1: join category from mapping on product_name (exact match)
dataframe_product = dataframe_product.merge(
    mapping[["product_name", "categories"]],
    on="product_name",
    how="left"
)

# Step 2: fallback — keyword matching on description for rows still without category
def find_category_by_keywords(param_description):
    """ Retourne la catégorie correspondante à une description de produit, en se basant sur les mots-clés définis dans le mapping """

    if pd.isna(param_description):
        return None

    description_upper = param_description.upper()
    for _, mapping_row in mapping.iterrows():
        brand_keyword = str(mapping_row["keywords_brands"]).upper()
        other_keywords = str(mapping_row["keywords_others"]).upper().split(";")

        if brand_keyword in description_upper and any(keyword.strip() in description_upper for keyword in other_keywords):
            return mapping_row["categories"]

    return None

mask_no_category = dataframe_product["categories"].isna()
dataframe_product.loc[mask_no_category, "categories"] = (
    dataframe_product.loc[mask_no_category, "description"]
    .apply(find_category_by_keywords)
)

# Step 3: When no category found after keyword matching, assign "Non catégorisé"
dataframe_product["categories"] = dataframe_product["categories"].fillna("Non catégorisé")

# Step 4: merge with database tables to get foreign keys (id_brand, id_category, id_unit, id_data_source)
dataframe_product = dataframe_product.merge(
    database_brand.rename(columns={"brand_name": "brand"}),
    on="brand",
    how="left"
)

dataframe_product = dataframe_product.merge(
    database_category.rename(columns={"category_name": "categories"}),
    on="categories",
    how="left"
)

dataframe_product = dataframe_product.merge(
    database_unit.rename(columns={"unit_name": "unit"}),
    on="unit",
    how="left"
)

dataframe_product = dataframe_product.merge(
    database_data_source.rename(columns={"data_source_name": "data_source"}),
    on="data_source",
    how="left"
)

In [ ]:
# Insert data into product table
insert_product = text("""
    INSERT INTO product
        (fk_id_brand, fk_id_category, fk_id_unit, fk_id_data_source,
        product_name, product_code, description)
    VALUES
        (:fk_id_brand, :fk_id_category, :fk_id_unit, :fk_id_data_source,
        :product_name, :product_code, :description)
""")

for _, row in dataframe_product.iterrows():
    connection_database.execute(insert_product, {
        "fk_id_brand":       int(row["id_brand"]),
        "fk_id_category":    int(row["id_category"]),
        "fk_id_unit":        int(row["id_unit"]),
        "fk_id_data_source": int(row["id_data_source"]),
        "product_name": row["product_name"] if pd.notna(row["product_name"]) else None,
        "product_code": row["product_code"] if pd.notna(row["product_code"]) else None,
        "description":  row["description"]  if pd.notna(row["description"])  else None,
    })

connection_database.commit()